<a href="https://colab.research.google.com/github/MaryumAkram16/Consumer-Complaint-Product-Classification/blob/main/complaint_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Consumer Complaint Product Classifier

Predicting the financial product category (Credit reporting, Mortgage, Debt
collection, etc.) from the free-text complaint narrative, trained on the CFPB
Consumer Complaint Database.

The raw file is ~8GB / 3.8M+ rows after filtering to complaints with a
narrative - too big to load normally on Colab's free 12GB RAM tier, so this
notebook uses lazy loading, a HashingVectorizer (no stored vocabulary), and
out-of-core batch training instead of the usual fit-on-everything-at-once
approach.

In [ ]:
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import gc
import os

## Load the data

Using `polars.scan_csv` for a lazy scan instead of pandas - only pulls in the
columns actually needed and filters out empty narratives before ever
materializing the full file in memory. A regular `pd.read_csv` on the whole
8GB file isn't feasible here.

In [ ]:
lazy_df = pl.scan_csv("/content/drive/MyDrive/complaints.csv")

df = (
    lazy_df
    .select([
        "Consumer complaint narrative",
        "Product",
        "Company response to consumer",
        "Timely response?"
    ])
    .filter(pl.col("Consumer complaint narrative").is_not_null())
    .collect()
)

print(f"Loaded shape: {df.shape}")

## Consolidating overlapping categories

The raw `Product` column has 21 values, but a lot of them are just the same
category renamed over time by CFPB's taxonomy changes (e.g. three different
labels all mean "Credit reporting"). Mapping these down to one clean label per
real category before training - otherwise the model would be trying to learn
distinctions that don't actually exist.

In [ ]:
category_mapping = {
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Credit reporting": "Credit reporting",
    "Debt collection": "Debt collection",
    "Debt or credit management": "Debt collection",
    "Checking or savings account": "Bank account or service",
    "Bank account or service": "Bank account or service",
    "Mortgage": "Mortgage",
    "Credit card": "Credit card or prepaid card",
    "Credit card or prepaid card": "Credit card or prepaid card",
    "Prepaid card": "Credit card or prepaid card",
    "Money transfer, virtual currency, or money service": "Money transfer or virtual currency",
    "Money transfers": "Money transfer or virtual currency",
    "Virtual currency": "Money transfer or virtual currency",
    "Student loan": "Student loan",
    "Vehicle loan or lease": "Vehicle loan or lease",
    "Payday loan, title loan, personal loan, or advance loan": "Payday, title, or personal loan",
    "Payday loan, title loan, or personal loan": "Payday, title, or personal loan",
    "Payday loan": "Payday, title, or personal loan",
    "Consumer Loan": "Payday, title, or personal loan",
}

df = df.with_columns(
    pl.col("Product").replace(category_mapping).alias("Product")
)

## Dropping the too-small category

`Other financial service` only has 292 rows - not enough for any model to
learn a reliable pattern from, so it's getting dropped rather than left in to
add noise.

In [ ]:
df = df.filter(pl.col("Product") != "Other financial service")

print(df["Product"].value_counts().sort("count", descending=True))

## Extract to numpy, free the DataFrame

Pulling just the two columns actually needed into plain numpy arrays and
dropping the polars DataFrame + lazy frame afterward - no reason to keep the
full thing in memory once the narratives and labels are out of it.

In [ ]:
narratives = df["Consumer complaint narrative"].to_numpy()
labels = df["Product"].to_numpy()

del df, lazy_df
gc.collect()

## Train/test split

Stratified so the class balance in train and test matches the full dataset -
some categories (like Payday loans) are a small fraction of the total, and a
non-stratified split risks skewing that further.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    narratives, labels, test_size=0.2, random_state=42, stratify=labels
)

del narratives, labels
gc.collect()

print(f"Type of y_train: {type(y_train)}")
print(f"Shape of y_train: {y_train.shape}")

## Vectorizer + class weights

Going with `HashingVectorizer` instead of `TfidfVectorizer` - TF-IDF needs to
build and store a full vocabulary before it can transform anything, which
risks running out of memory at this row count. HashingVectorizer maps tokens
straight to a fixed number of hash buckets with no stored vocabulary at all,
so memory stays flat no matter how big the dataset gets.

Class weights computed manually up front since `SGDClassifier.partial_fit()`
doesn't support `class_weight='balanced'` directly - that requires knowing the
full class distribution ahead of time, which partial_fit by design doesn't
have access to mid-training.

In [ ]:
vectorizer = HashingVectorizer(
    n_features=2**19,
    stop_words="english",
    ngram_range=(1, 2),
    alternate_sign=False
)

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print(class_weight_dict)

## Train out-of-core

The full vectorized training set would be a huge sparse matrix if built all at
once - instead, training in batches of 30,000 rows: vectorize the batch, fit,
discard, repeat. `SGDClassifier` (hinge loss - equivalent to a linear SVM)
supports this directly via `partial_fit`. Running 3 epochs over the data with
a fresh shuffle each epoch.

In [ ]:
clf = SGDClassifier(
    loss="hinge",
    class_weight=class_weight_dict,
    random_state=42
)

batch_size = 30_000
n_samples = len(X_train)
n_epochs = 3

for epoch in range(1, n_epochs + 1):
    print(f"\n=== Epoch {epoch}/{n_epochs} ===")
    indices = np.random.RandomState(epoch).permutation(n_samples)

    for i, start in enumerate(range(0, n_samples, batch_size)):
        end = min(start + batch_size, n_samples)
        batch_idx = indices[start:end]

        X_batch = vectorizer.transform(X_train[batch_idx])
        y_batch = y_train[batch_idx]

        clf.partial_fit(X_batch, y_batch, classes=classes)
        del X_batch

        if i % 20 == 0:
            gc.collect()
            print(f"  Epoch {epoch}: trained on {end}/{n_samples} rows")

    gc.collect()
    print(f"Epoch {epoch} complete.")

print("\nAll epochs complete.")

## Evaluate

Same out-of-core approach for evaluation - predicting in batches instead of
transforming the entire test set into one matrix at once.

In [ ]:
y_pred = []
batch_size_eval = 30_000
for start in range(0, len(X_test), batch_size_eval):
    end = min(start + batch_size_eval, len(X_test))
    X_batch = vectorizer.transform(X_test[start:end])
    y_pred.extend(clf.predict(X_batch))
    del X_batch

y_pred = np.array(y_pred)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

## Confusion matrix

Normalizing by actual class (row-wise) instead of showing raw counts - Credit
reporting has way more rows than everything else, so raw counts would just
show one giant dominant row and hide the more interesting confusion patterns
in the smaller categories.

In [ ]:
labels_sorted = sorted(np.unique(y_test))

cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    xticklabels=labels_sorted,
    yticklabels=labels_sorted,
    cmap='Blues',
    cbar_kws={'label': 'Proportion of actual class'}
)
plt.xlabel('Predicted Category')
plt.ylabel('Actual Category')
plt.title('Confusion Matrix — Complaint Product Classification (Normalized)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('complaint-product-classification.PNG', dpi=150, bbox_inches='tight')
plt.show()

## Try it on a sample complaint

In [ ]:
def predict_category(text, clf, vectorizer):
    X = vectorizer.transform([text])
    return clf.predict(X)[0]

sample = "I have been trying to get my credit report corrected for months and no one responds."
print(predict_category(sample, clf, vectorizer))

## Save everything

Saving with compression directly this time (`compress=3`) rather than saving
twice - no reason to write the uncompressed version to disk first when the
compressed one is what's actually getting downloaded and deployed.

In [ ]:
joblib.dump(clf, 'complaint_product_classifier.pkl', compress=3)
joblib.dump(vectorizer, 'hashing_vectorizer.pkl', compress=3)

print(os.path.getsize('complaint_product_classifier.pkl') / 1e6, "MB")
print(os.path.getsize('hashing_vectorizer.pkl') / 1e6, "MB")

In [ ]:
from google.colab import files

files.download('complaint_product_classifier.pkl')
files.download('hashing_vectorizer.pkl')
files.download('complaint-product-classification.PNG')